In [1]:
import json
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
import spacy

from dap_job_quality.utils.keyword_search_patterns import keywords
from dap_job_quality.getters.ojo_getters import get_ojo_sample
from dap_job_quality.utils.spacy_keyword_search import get_matches, get_spans
from dap_job_quality.utils.text_cleaning import clean_text

from dap_job_quality.getters.data_getters import load_s3_jsonl
from dap_job_quality.getters.labelled_data import get_labelled_job_sentences
from dap_job_quality.utils import prodigy_data_utils as pdu

from dap_job_quality import BUCKET_NAME, PROJECT_DIR, config

model = SentenceTransformer("all-MiniLM-L6-v2")

/Users/rosie.oxbury/miniconda3/envs/dap_job_quality/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2024-05-02 15:02:21,500 - botocore.credentials - INFO - Found credentials in shared credentials file: ~/.aws/credentials
2024-05-02 15:02:22,244 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2
2024-05-02 15:02:22,525 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device: cpu


In [2]:
nlp = spacy.load("en_core_web_sm")

SEED = config["seed"]

def get_negative_example_sentences(df):
    # Find all unique texts
    unique_texts = df["text"].unique()

    # Split unique texts into sentences
    all_sentences_from_text = set()
    for text in unique_texts:
        doc = nlp(text)
        all_sentences_from_text.update([sent.text for sent in doc.sents])

    # Set of sentences already in the 'sentence' column
    existing_sentences = set(df["sentence"])

    # Find sentences that are not in the 'sentence' column
    new_sentences = all_sentences_from_text - existing_sentences

    return new_sentences


def filter_job_ads(labelled_df):
    job_ids = labelled_df["id"].unique()

    # skip the first 10 job ads - we didn't know what we were labelling at that point
    target_ids = job_ids[10:]

    labelled_df_clean = labelled_df[labelled_df["id"].isin(target_ids)]
    # get rid of empty spans
    labelled_df_clean = labelled_df_clean[labelled_df_clean["span"] != ""]
    return labelled_df_clean

In [3]:
labelled_sents = get_labelled_job_sentences()[0]

labelled_data = pdu.get_spans_and_sentences(labelled_sents)

labelled_df = pd.DataFrame(columns=["span", "sent", "text", "job_id"])

for key in labelled_data.keys():
    temp_df = pd.DataFrame(labelled_data[key])
    temp_df["id"] = int(key)
    labelled_df = pd.concat([labelled_df, temp_df])

labelled_df = labelled_df.drop(["job_id"], axis=1)

labelled_df_clean = filter_job_ads(labelled_df)

labelled_df_clean["sentence"] = labelled_df_clean["sent"].apply(lambda x: x.text)

2024-05-02 15:02:24,283 - dap_job_quality - INFO - File job_quality/prodigy/binary_classifier_labelled_data/20240416/job_sentences_labelled_20240416.jsonl downloaded from open-jobs-lake to /Users/rosie.oxbury/Documents/git_repos/dap_job_quality/inputs/labelled/job_sentences_labelled_20240416.jsonl


In [4]:
negative_sentences = get_negative_example_sentences(labelled_df_clean)
negative_df = pd.DataFrame(list(negative_sentences))
negative_df.head()

,0
0,Get to know the local market to support plans ...
1,"Support the direct teams with their meetings, ..."
2,At James Andrews Recruitment Solutions we try ...
3,A Bristol based telecommunications firm need a...
4,If this Account Manager role sounds like the i...


In [5]:

negative_df["label"] = 0
negative_df.columns = ["sentence", "label"]

positive_df = labelled_df_clean[["sentence"]]
positive_df["label"] = 1

training_ml_df = pd.concat([positive_df, negative_df])

/var/folders/x3/w5p1j0m5745bvc10jk_w36900000gn/T/ipykernel_53571/1198652558.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  positive_df["label"] = 1


In [6]:
# Splitting the dataset into training, validation, and test sets
X_train, X_temp, y_train, y_temp = train_test_split(
        training_ml_df.drop(["label"], axis=1),
        training_ml_df["label"],
        test_size=0.4,
        random_state=SEED,
    )
X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=SEED
    )

In [7]:
input_df = pd.concat([X_train, y_train], axis=1)
input_df = input_df[input_df['label']==1]

In [8]:
len(input_df)

119

In [10]:
input_sentences = input_df['sentence'].tolist()

In [11]:
lookup = pd.read_csv(PROJECT_DIR / "inputs/keyword_lookup - v3.csv")
lookup.head(20)

,dimension,subcategory,target_phrase,Notes
0,pay and benefits,COMP,pension,NaN
1,pay and benefits,COMP,bonus,NaN
2,pay and benefits,COMP,salary,NaN
3,pay and benefits,COMP,compensation,NaN
4,pay and benefits,COMP,pay,NaN
5,pay and benefits,COMP,per annum,NaN
6,pay and benefits,COMP,overtime,NaN
7,pay and benefits,LEAVE,leave,NaN
8,pay and benefits,LEAVE,holiday,NaN
9,pay and benefits,LEAVE,vacation,NaN


In [12]:
targets = lookup['target_phrase'].tolist()

In [13]:
target_embeddings = model.encode(targets, show_progress_bar=True)

Batches: 100%|██████████| 2/2 [00:00<00:00,  6.65it/s]


In [14]:
lookup['embeddings'] = target_embeddings.tolist()

In [15]:
def get_n_most_similar_phrases(input_sentence, 
                                 lookup,
                                 model,
                                 n: int = 3):
    most_similar_phrases = {}
    
    input_embedding = model.encode(input_sentence)
    
    similarities = [cosine_similarity([input_embedding], [embed])[0][0] for embed in lookup['embeddings'].apply(pd.Series).values]
    
    top_indices = np.argsort(similarities)[::-1][:n]
    
    similar_phrases = lookup.iloc[top_indices]
    similar_phrases['similarity'] = [similarities[i] for i in top_indices]
    
    most_similar_phrases[input_sentence] = similar_phrases[['dimension', 'subcategory', 'target_phrase', 'similarity']]
        
    return most_similar_phrases

In [ ]:
# for _, row in X_train.iterrows():
#     output = get_n_most_similar_phrases(row['span'], lookup, model, 1)
#     output['span'] = row['span']
    

In [16]:
for sentence in input_sentences[0:20]:
    print(f"Sentence: {sentence}")
    print(get_n_most_similar_phrases(sentence, lookup, model, 3))

Sentence: Location  London, Piccadilly Circus Role  Graduate Researcher - Executive Search Basic  £30,000 - £35,000K (+ bonuses of up to 90%)


Batches: 100%|██████████| 1/1 [00:00<00:00, 34.02it/s]
/var/folders/x3/w5p1j0m5745bvc10jk_w36900000gn/T/ipykernel_53571/2912512558.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  similar_phrases['similarity'] = [similarities[i] for i in top_indices]


{'Location  London, Piccadilly Circus Role  Graduate Researcher - Executive Search Basic  £30,000 - £35,000K (+ bonuses of up to 90%)':                         dimension subcategory   target_phrase  similarity
2                pay and benefits        COMP          salary    0.354010
5                pay and benefits        COMP       per annum    0.308278
32  job design and nature of work      CAREER  career advance    0.286703}
Sentence: Job Title  2nd Line IT Support Engineer Location  Chiswick, West London Salary  Up to £28,000 depending on experience Job Type  Full Time, Permanent Hours  37.5 hours per week (Covering 10 00 - 18 30 Mon - Fri.


Batches: 100%|██████████| 1/1 [00:00<00:00, 36.67it/s]
/var/folders/x3/w5p1j0m5745bvc10jk_w36900000gn/T/ipykernel_53571/2912512558.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  similar_phrases['similarity'] = [similarities[i] for i in top_indices]


{'Job Title  2nd Line IT Support Engineer Location  Chiswick, West London Salary  Up to £28,000 depending on experience Job Type  Full Time, Permanent Hours  37.5 hours per week (Covering 10 00 - 18 30 Mon - Fri.':             dimension subcategory           target_phrase  similarity
2    pay and benefits        COMP                  salary    0.357689
5    pay and benefits        COMP               per annum    0.298059
12  work life balance  FLEX_HOURS  flexible working hours    0.248333}
Sentence: Structured career development opportunities Brand new fully kitted out offices, minutes from the subway station making it a dream location for ALL!


...

{'Telemarketing Executive Pay  £23,000 per annum Location  Ferndown Industrial Estate, Wimborne Hours  Monday - Friday, up to 37 ½ hours per week Contract Type  Temporary to Permanent Our well-established client is looking for a Telemarketing Executive to assist with their current period of growth.':             dimension subcategory target_phrase  similarity
2    pay and benefits        COMP        salary    0.323171
5    pay and benefits        COMP     per annum    0.304795
16  work life balance  FLEX_HOURS     job share    0.279222}
Sentence: Basic Salary  £24,000 - On Target Earnings of £40,000 in your first 12 months!Our fantastic Phaidon training with our on-site dedicated Learning and Development team - ensuring you get the support you need!A fun, dynamic working environment in our amazing London HQ - right next to the Bank of England.


Batches: 100%|██████████| 1/1 [00:00<00:00, 43.74it/s]
/var/folders/x3/w5p1j0m5745bvc10jk_w36900000gn/T/ipykernel_53571/2912512558.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  similar_phrases['similarity'] = [similarities[i] for i in top_indices]


{'Basic Salary  £24,000 - On Target Earnings of £40,000 in your first 12 months!Our fantastic Phaidon training with our on-site dedicated Learning and Development team - ensuring you get the support you need!A fun, dynamic working environment in our amazing London HQ - right next to the Bank of England.':           dimension subcategory target_phrase  similarity
2  pay and benefits        COMP        salary    0.481758
5  pay and benefits        COMP     per annum    0.317547
4  pay and benefits        COMP           pay    0.287203}
Sentence: Flexible working, exceptional holidays and joining a defined benefit  pension scheme are on offer.


Batches: 100%|██████████| 1/1 [00:00<00:00, 85.62it/s]
/var/folders/x3/w5p1j0m5745bvc10jk_w36900000gn/T/ipykernel_53571/2912512558.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  similar_phrases['similarity'] = [similarities[i] for i in top_indices]


{'Flexible working, exceptional holidays and joining a defined benefit  pension scheme are on offer.':             dimension subcategory             target_phrase  similarity
0    pay and benefits        COMP                   pension    0.561155
15  work life balance  FLEX_HOURS  flexible working options    0.469663
12  work life balance  FLEX_HOURS    flexible working hours    0.468270}
